# 인용 쌍으로 임베딩 파인튜닝 (Colab T4)

**이 노트북이 답하는 질문 두 개**

1. `bge-base-en-v1.5`는 `text-embedding-3-large`와 **동등합니다**(recall@10 0.1909 대 0.2019,
   95% CI [-0.0250, +0.0032]로 0 포함). 인용 쌍으로 파인튜닝하면 **넘는가?**
2. 그래프에서 뽑은 **hard negative**는 선형 어댑터에서 기각됐습니다(정규화 6개 수준 전부에서
   이기지 못함). 용량 있는 모델에서는 **작동하는가?**

두 질문 다 GPU 없이는 답할 수 없습니다. 개발 PC에는 NVIDIA가 없어(Intel Iris Xe) 37,854편
인코딩이 CPU로 3.5시간이고 파인튜닝은 비현실적입니다.

**범위.** 오프라인 측정까지입니다. 앱에 넣으려면 ① 37,854편 재임베딩 ② 질의 시 로컬 모델 추론이
필요하고, ②가 torch를 앱 의존성으로 끌어옵니다(현재 핵심 경로 259MB, Streamlit 보장 690MB).
배포는 별도 판단입니다.

**실행 순서.** `SMOKE = True`로 한 번 돌려 코드가 도는지 보고, 그다음 `False`로 본 실행을 하세요.
SMOKE 수치는 말뭉치를 줄여 재므로 **결과로 읽지 마세요.**

In [ ]:
# 설정: SMOKE=True면 작은 부분만으로 코드가 도는지 확인합니다(CPU도 가능).
SMOKE = True

BRANCH = "main"
REPO_URL = "https://github.com/rndigkwk/arxiv_graph_RAG.git"
DATA_REPO = "<사용자>/arxiv-graphrag-colab"   # HF 비공개 dataset repo. 아래에서 올립니다.
BASE_MODEL = "BAAI/bge-base-en-v1.5"

# 제목+초록 토큰 중앙값이 258입니다. 128이나 256으로 자르면 논문 절반이 잘려
# 모델이 아니라 절단을 재게 됩니다.
MAX_SEQ_LENGTH = 512

BATCH_SIZE = 32          # T4 16GB 기준. OOM이 나면 16으로 내리세요.
LEARNING_RATE = 2e-5
EPOCHS = 1
SEED = 20261015
TOP_K = 10
HF_REPO_PREFIX = "bge-arxiv-citation"   # <사용자>/bge-arxiv-citation-inbatch, -hard

SMOKE_CORPUS = 3000     # SMOKE에서 후보로 쓸 논문 수
SMOKE_PAIRS = 400
SMOKE_QUERIES = 40

In [ ]:
import sys

print("python", sys.version.split()[0])
try:
    import torch

    print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except ImportError:
    print("torch 아직 없음 (다음 셀에서 설치)")

In [ ]:
# 저장소와 데이터를 받습니다. 로컬에서 SMOKE로 돌릴 때는 저장소 루트에서 실행되므로 clone을 건너뜁니다.
import os
from pathlib import Path

ON_COLAB = os.path.exists("/content")
if ON_COLAB:
    %cd /content
    !rm -rf arxiv_graph_RAG
    !git clone --depth 1 --branch {BRANCH} {REPO_URL}
    %cd /content/arxiv_graph_RAG
    !pip install -q "sentence-transformers[train]>=5.1" "datasets>=2.19"

# Hugging Face 토큰: VS Code Colab 확장에서는 Colab 비밀 값을 못 읽어 직접 입력합니다.
token = None
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    pass
if token is None:
    from getpass import getpass

    token = getpass("Hugging Face 토큰 (쓰기 권한): ")

DATA_DIR = Path("exports/colab")
if not (DATA_DIR / "manifest.json").exists():
    from huggingface_hub import snapshot_download

    snapshot_download(
        DATA_REPO, repo_type="dataset", token=token,
        local_dir=".", allow_patterns=["exports/colab/*"],
    )

import importlib.metadata as metadata

for package in ("sentence-transformers", "transformers", "torch", "datasets", "accelerate"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "없음")

In [ ]:
import gzip
import json
import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)


def read_jsonl(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
corpus_rows = read_jsonl(DATA_DIR / "corpus.jsonl.gz")
train_rows = read_jsonl(DATA_DIR / "train_pairs.jsonl.gz")
eval_rows = read_jsonl(DATA_DIR / "eval_queries.jsonl.gz")
baseline_rows = read_jsonl(DATA_DIR / "baseline_recall.jsonl.gz")

print("snapshot", manifest["snapshot_version"], "| papers sha256",
      manifest["snapshot_papers_sha256"][:16])
print("corpus", len(corpus_rows), "| train pairs", len(train_rows),
      "| eval queries", len(eval_rows))
print("baseline", manifest["baseline"]["model"], "recall@10",
      manifest["baseline"]["recall_at_10"])
print(manifest["max_seq_length_note"])

In [ ]:
# SMOKE에서는 말뭉치·쌍·질의를 줄입니다. 후보 수가 달라지므로 SMOKE 수치는 비교에 쓸 수 없습니다.
if SMOKE:
    keep_ids = {row["anchor"] for row in eval_rows[:SMOKE_QUERIES]}
    for row in eval_rows[:SMOKE_QUERIES]:
        keep_ids.update(row["gold"])
    extra = [row["arxiv_id"] for row in corpus_rows if row["arxiv_id"] not in keep_ids]
    keep_ids.update(extra[: max(0, SMOKE_CORPUS - len(keep_ids))])

    corpus_rows = [row for row in corpus_rows if row["arxiv_id"] in keep_ids]
    eval_rows = eval_rows[:SMOKE_QUERIES]
    train_rows = [
        row for row in train_rows
        if {row["anchor"], row["positive"], row["hard_negative"]} <= keep_ids
    ][:SMOKE_PAIRS]
    print("SMOKE: corpus", len(corpus_rows), "pairs", len(train_rows),
          "queries", len(eval_rows))
    print("경고: 후보 수를 줄였으므로 아래 수치를 결과로 읽지 마세요.")

text_of = {row["arxiv_id"]: row["text"] for row in corpus_rows}
corpus_ids = [row["arxiv_id"] for row in corpus_rows]
position = {arxiv_id: index for index, arxiv_id in enumerate(corpus_ids)}
baseline_recall = {row["anchor"]: row["recall_at_10"] for row in baseline_rows}

queries = [
    (row["anchor"], [paper for paper in row["gold"] if paper in position])
    for row in eval_rows
    if row["anchor"] in position
]
queries = [(anchor, gold) for anchor, gold in queries if gold]
print("사용할 질의", len(queries))

In [ ]:
import torch
from sentence_transformers import SentenceTransformer


def encode_all(model, batch_size=64):
    vectors = model.encode(
        [text_of[arxiv_id] for arxiv_id in corpus_ids],
        batch_size=batch_size, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    )
    return np.asarray(vectors, dtype=np.float32)


def per_query_recall(vectors, top_k=TOP_K):
    """질의 논문 자신을 후보에서 빼고 전체 말뭉치를 순위 매깁니다."""
    rows = np.stack([vectors[position[anchor]] for anchor, _ in queries])
    similarity = rows @ vectors.T
    for index, (anchor, _) in enumerate(queries):
        similarity[index, position[anchor]] = -np.inf
    ranked = np.argpartition(-similarity, top_k, axis=1)[:, :top_k]
    recalls, hits, reciprocal = [], [], []
    for index, (_, gold) in enumerate(queries):
        order = ranked[index][np.argsort(-similarity[index, ranked[index]])]
        retrieved = [corpus_ids[spot] for spot in order]
        found = [rank for rank, paper in enumerate(retrieved, start=1) if paper in set(gold)]
        recalls.append(len(set(gold) & set(retrieved)) / len(gold))
        hits.append(float(bool(found)))
        reciprocal.append(1.0 / found[0] if found else 0.0)
    return np.asarray(recalls), np.asarray(hits), np.asarray(reciprocal)


def metrics(vectors):
    recalls, hits, reciprocal = per_query_recall(vectors)
    return {
        "queries": len(queries),
        f"recall_at_{TOP_K}": round(float(recalls.mean()), 4),
        f"hit_at_{TOP_K}": round(float(hits.mean()), 4),
        "mrr": round(float(reciprocal.mean()), 4),
    }, recalls


def paired_bootstrap(baseline, candidate, samples=2000):
    """질의를 쌍으로 재표본합니다. 평균이 질의 몇 건 차이일 때 그것이 측정인지 가릅니다."""
    difference = candidate - baseline
    generator = np.random.default_rng(SEED)
    indices = generator.integers(0, len(difference), size=(samples, len(difference)))
    means = difference[indices].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return {
        "mean_difference": round(float(difference.mean()), 4),
        "ci95_low": round(float(low), 4),
        "ci95_high": round(float(high), 4),
        "excludes_zero": bool(low > 0.0 or high < 0.0),
    }


openai_recall = np.asarray([baseline_recall[anchor] for anchor, _ in queries])
print("OpenAI 기준선 recall@10 (이 질의 집합)", round(float(openai_recall.mean()), 4))

In [ ]:
# 튜닝 전 기준선. SMOKE가 아니라면 recall@10이 0.1909 근처여야 합니다.
# 크게 다르면 토크나이저·정규화·질의 집합 중 하나가 어긋난 것이고, 그때는 비교가 성립하지 않습니다.
untuned = SentenceTransformer(BASE_MODEL)
untuned.max_seq_length = MAX_SEQ_LENGTH
untuned_vectors = encode_all(untuned)
untuned_metrics, untuned_recall = metrics(untuned_vectors)
print("untuned", untuned_metrics)
if not SMOKE:
    gap = abs(untuned_metrics[f"recall_at_{TOP_K}"] - 0.1909)
    print("기대값 0.1909과의 차이", round(gap, 4),
          "-> 재현됨" if gap < 0.01 else "-> 어긋남, 비교 전에 원인을 찾으세요")

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss


def build_dataset(with_hard_negative):
    """하드 네거티브를 열로 주면 MNR이 그것을 배치 안 후보에 더합니다."""
    columns = {
        "anchor": [text_of[row["anchor"]] for row in train_rows],
        "positive": [text_of[row["positive"]] for row in train_rows],
    }
    if with_hard_negative:
        columns["negative"] = [text_of[row["hard_negative"]] for row in train_rows]
    return Dataset.from_dict(columns)


def finetune(variant, with_hard_negative):
    model = SentenceTransformer(BASE_MODEL)
    model.max_seq_length = MAX_SEQ_LENGTH
    arguments = SentenceTransformerTrainingArguments(
        output_dir=f"/tmp/out-{variant}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        logging_steps=50,
        save_strategy="no",
        report_to=[],
    )
    trainer = SentenceTransformerTrainer(
        model=model,
        args=arguments,
        train_dataset=build_dataset(with_hard_negative),
        loss=MultipleNegativesRankingLoss(model),
    )
    trainer.train()
    model.save(f"/tmp/out-{variant}")
    return model


models = {}
models["inbatch"] = finetune("inbatch", with_hard_negative=False)
models["hard"] = finetune("hard", with_hard_negative=True)
print("학습 완료", list(models))

In [ ]:
results = {
    "text-embedding-3-large": {
        "recall_at_10": round(float(openai_recall.mean()), 4),
        "source": "스냅샷에서 내보낸 질의별 점수",
    },
    f"{BASE_MODEL} (untuned)": {
        **untuned_metrics,
        "vs_openai": paired_bootstrap(openai_recall, untuned_recall),
    },
}
for variant, model in models.items():
    vectors = encode_all(model)
    row, recall = metrics(vectors)
    results[f"{BASE_MODEL} ({variant})"] = {
        **row,
        "vs_openai": paired_bootstrap(openai_recall, recall),
        "vs_untuned": paired_bootstrap(untuned_recall, recall),
    }

report = {
    "smoke": SMOKE,
    "snapshot_version": manifest["snapshot_version"],
    "snapshot_papers_sha256": manifest["snapshot_papers_sha256"],
    "base_model": BASE_MODEL,
    "max_seq_length": MAX_SEQ_LENGTH,
    "hyperparameters": {
        "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "seed": SEED,
    },
    "corpus": len(corpus_ids),
    "train_pairs": len(train_rows),
    "queries": len(queries),
    "results": results,
}
print(json.dumps(report, ensure_ascii=False, indent=2))
Path("/tmp/colab_finetune_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

## 읽는 법

**질문 1 — 파인튜닝이 API 모델을 넘는가.** `vs_openai`의 `excludes_zero`를 보세요. `True`이고
`mean_difference`가 양수면 넘은 것입니다. 구간이 0을 포함하면 **넘지 못했거나 측정되지
않았습니다** — 평균이 올라갔다는 것만으로는 부족합니다.

**질문 2 — hard negative가 작동하는가.** `hard`와 `inbatch`의 `vs_untuned`를 비교하세요.
`hard`가 더 크면 그래프에서 뽑은 음성이 기여한 것이고, 선형 어댑터에서의 기각이 **모델 용량
문제**였다는 뜻입니다. 둘이 같거나 `hard`가 낮으면 신호 자체가 약한 것입니다.

**`untuned`가 0.1909을 재현하지 않으면 그 아래 수치는 모두 무의미합니다.** 비교의 전제가
"같은 과제·같은 질의·같은 후보"이기 때문입니다. 차이가 크면 먼저 원인을 찾으세요.

In [ ]:
# 모델과 보고서를 Hugging Face 비공개 저장소에 올립니다(처음에 입력한 토큰, 쓰기 권한 필요).
if not SMOKE:
    from huggingface_hub import HfApi

    api = HfApi(token=token)
    user = api.whoami()["name"]
    for variant in models:
        repo = f"{user}/{HF_REPO_PREFIX}-{variant}"
        api.create_repo(repo, private=True, exist_ok=True)
        api.upload_folder(repo_id=repo, folder_path=f"/tmp/out-{variant}")
        api.upload_file(
            path_or_fileobj="/tmp/colab_finetune_report.json",
            path_in_repo="colab_finetune_report.json", repo_id=repo,
        )
        print("uploaded", repo)
del token

## 결과를 저장소로 가져오기

`colab_finetune_report.json`을 `reports/embedding_finetune_colab.json`으로 커밋하면 다른
측정들과 같은 자리에 남습니다. 모델 가중치는 커밋하지 않습니다 — HF 비공개 저장소에 두고
보고서만 가져옵니다.